# Context Management for AI Agents in Prectice

_Author: [Vrunda Gadesha](https://www.ibm.com/think/author/vrunda-gadesha)_

This notebook is the runnable companion to [`context_management.md`](./context_management.md). It demonstrates all nine context management strategies using **IBM Granite** on watsonx.ai so you can see real LLM behaviour.

The context window is an agent's entire working memory. Every strategy below addresses one or more of the four failure modes identified in the blog:

| Failure mode | Root cause | Strategies Resolved |
| :--- | :--- | :--- |
| **Context Poisoning** | Hallucinations re-enter as ground truth |Context Pruning|
| **Context Distraction** | Too much history overwhelms the model |FIFO / Rolling Window, Compaction / Summarization, Structured Note-Taking + Filesystem, Semantic Compression, Sub-agent Architectures|
| **Context Confusion** | Irrelevant tokens pull the model off-target | Dynamic Tool Selection, Context Pruning, Semantic Compression, RAG / Vector Retrieval |
| **Context Clash** | Contradictory information accumulates | Structured Note-Taking + Filesystem, Sub-agent Architectures |

## Setup & Installation

You can run this notebook in [Colab](https://colab.research.google.com/) or locally. To avoid Python package conflicts, we recommend a [virtual environment](https://docs.python.org/3/library/venv.html).

### Install dependencies


In [ ]:
%pip install -q \
    "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    langchain-core \
    langchain_ibm \
    sentence-transformers \
    numpy

## Import Packages

In [ ]:
from __future__ import annotations

import json
import math
import re
import shutil
from collections import Counter
from pathlib import Path
from typing import TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

from langchain.chat_models import init_chat_model
from langchain_core.utils.utils import convert_to_secret_str
from ibm_granite_community.notebook_utils import get_env_var

import warnings
from ibm_watsonx_ai.wml_resource import WatsonxAPIWarning

warnings.filterwarnings("ignore", category=WatsonxAPIWarning)


### Connect to the Granite model on watsonx.ai

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for setup instructions.

You need these environment variables (in a `.env` file or exported in your shell):
- `WATSONX_URL` — your watsonx.ai endpoint
- `WATSONX_APIKEY` — your API key
- `WATSONX_PROJECT_ID` — your project ID


In [ ]:
MODEL_ID = "ibm/granite-4-h-small"

llm = init_chat_model(
    model=MODEL_ID,
    model_provider="ibm",
    url=convert_to_secret_str(get_env_var("WATSONX_URL")),
    apikey=convert_to_secret_str(get_env_var("WATSONX_APIKEY")),
    project_id=get_env_var("WATSONX_PROJECT_ID"),
    params={"temperature": 0, "max_new_tokens": 512},
)

# Quick sanity check
from langchain_core.messages import HumanMessage
ping = llm.invoke([HumanMessage(content="Reply with exactly: ready")])
print("Model response:", ping.content)

## Strategy 1 — FIFO / Rolling Window

**Goal:** Keep the active prompt small by capping history length to the most recent $N$ messages.

**How it works:** Standard chat models remember past context because we resend the message history with every new turn. A Rolling Window (First-In, First-Out) keeps only the system prompt plus the last $N$ messages. Older turns are automatically dropped off the conveyor belt.Let's define a helper function `apply_fifo_window()` that preserves our `SystemMessage` (so the model remembers its assigned persona) while trimming the remaining chat history to our desired window size.


### Lightweight Text Helper Functions

The three functions below (`simple_tokenize`, `keyword_score`, and `cosine_similarity`) are Python text utilities used across notebook as per the requirements. 

In [ ]:
# Lightweight text helpers

# This helper function convert the string into lowercase, it finds every sequence of letters and digits, discarding punctuation, spaces, and special characters and then store it in list format as one word and one token. 
def simple_tokenize(text: str) -> list[str]:
    """Lowercase and split text into alphanumeric tokens.

        Note: This is a naive implementation for quick comparison. In production or for real token metrics, use the model tokenizer endpoint from Watsonx.ai.
    """
    return re.findall(r"[a-z0-9]+", text.lower())

# This helper function counts how many words from the query also appear in the text (exact token overlap).
def keyword_score(query: str, text: str) -> int:
    """Count how many query tokens appear in the target text (used in Strategy 4)."""
    query_terms = set(simple_tokenize(query))
    text_terms  = set(simple_tokenize(text))
    return len(query_terms & text_terms)

# This helper function measures how directionally similar two texts are by comparing their word frequency vectors, returning a float between 0.0 (no overlap) and 1.0 (identical).
def cosine_similarity(query: str, text: str) -> float:
    """Compute bag-of-words cosine similarity between two strings (used in Strategy 8).

        Note: Zero-dependency implementation for the codebook. Production workflows typically utilize 
        optimized vectorizers from libraries like scikit-learn, while agent frameworks 
        such as LangGraph offer built-in retrieval and state-routing strategies.
    """
    left  = Counter(simple_tokenize(query))
    right = Counter(simple_tokenize(text))
    shared    = set(left) & set(right)
    numerator = sum(left[t] * right[t] for t in shared)
    left_norm  = math.sqrt(sum(v * v for v in left.values()))
    right_norm = math.sqrt(sum(v * v for v in right.values()))
    if not left_norm or not right_norm:
        return 0.0
    return numerator / (left_norm * right_norm)

### Build the Conversation History & Define FIFO Trimming

In a multi-turn conversation, past messages accumulate quickly. First, we generate a multi-turn conversation with Granite about planning a trip to Tokyo. Then, we write a `apply_fifo_window()` function that keeps the system prompt intact while capping the active chat history to only the $N$ most recent turns.

In [ ]:
# =====================================================================
# Strategy 1 — FIFO / Rolling Window (Part 1: Build & Define)
# =====================================================================

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# 1. Compact helper function to print message history cleanly
def print_compact_conversation(messages: list, title: str = "CONVERSATION HISTORY"):
    print(f"\n--- {title} ({len(messages)} messages) ---")
    for idx, msg in enumerate(messages, 1):
        if isinstance(msg, SystemMessage):
            tag = "SYS"
        elif isinstance(msg, HumanMessage):
            tag = "USER"
        else:
            tag = "BOT"
            
        # Clean up inner newlines so each turn stays on a single line
        clean_text = msg.content.replace("\n", " ").strip()
        print(f"{idx}. [{tag}]: {clean_text}")


# 2. Set up persona and user turns
system_prompt = SystemMessage(content="You are a helpful travel planning assistant. Answer concisely.")

turns = [
    "I want to plan a 7-day trip to Tokyo. What season do you recommend?",
    "What is the typical budget range per day for a mid-range traveller?",
    "Which neighbourhoods are best for first-time visitors?",
    "Can you suggest two must-see cultural sites?",
    "What is the easiest way to get from Narita airport to central Tokyo?",
]


# 3. Generate multi-turn history with IBM Granite
conversation: list = [system_prompt]

for user_text in turns:
    conversation.append(HumanMessage(content=user_text))
    response = llm.invoke(conversation)
    conversation.append(response)


# 4. Define the FIFO (Rolling Window) trimming logic — pair-aware
def apply_fifo_window(messages: list, window_size: int = 3) -> list:
    """
    Preserves the SystemMessage at index 0 (if present) and keeps only the
    last `window_size` complete USER+BOT pairs so the model never sees a
    BOT reply without the USER question that prompted it.

    `window_size` counts pairs, not individual messages:
      window_size=3  ->  up to 6 messages (3 USER + 3 BOT)
    """
    if not messages:
        return []

    if isinstance(messages[0], SystemMessage):
        system_msg = [messages[0]]
        chat_history = messages[1:]
    else:
        system_msg = []
        chat_history = messages

    # Build a list of complete (USER, BOT) pairs from the chat history.
    pairs = []
    i = 0
    while i < len(chat_history) - 1:
        if isinstance(chat_history[i], HumanMessage) and isinstance(chat_history[i + 1], AIMessage):
            pairs.append((chat_history[i], chat_history[i + 1]))
            i += 2
        else:
            i += 1  # skip malformed / interleaved messages

    # Keep the last `window_size` complete pairs
    recent_pairs = pairs[-window_size:]
    trimmed_chat = [msg for pair in recent_pairs for msg in pair]

    # If there is a trailing lone HumanMessage (an in-progress turn), append it
    if chat_history and isinstance(chat_history[-1], HumanMessage):
        trimmed_chat.append(chat_history[-1])

    return system_msg + trimmed_chat


# 5. Display the full generated conversation
print_compact_conversation(conversation, title="FULL CONVERSATION HISTORY GENERATED WITH GRANITE")
print("\nFIFO Trimming function ready.")

### Inspect the Trimmed Context Window

Now we pass our 11-message conversation through `apply_fifo_window()` with `window_size = 3`. This shows how the window drops older context (like budget and season discussions) while retaining the system prompt and the latest **complete** USER+BOT pairs.


**Why pair-wise retention matters**

A naive message-count window (e.g. "keep last 3 messages") can produce a context like this:

```
[SYS]:  You are a helpful travel planning assistant.
[BOT]:  Two must-see cultural sites are Senso-ji Temple and Meiji Shrine.  ← orphaned!
[USER]: What is the easiest way to get from Narita airport to central Tokyo?
[BOT]:  The easiest way is by taking the Narita Express (N'EX).
```

Message 2 is a BOT reply with **no USER question before it**. The model has no idea what was asked — it may treat the orphaned answer as an unprompted assertion, hallucinate a missing question, or drift off-topic. Keeping whole USER→BOT pairs guarantees every answer in context has its question alongside it.

In [ ]:
# --- Part 2: Inspect Context Before vs. After Trimming ---

WINDOW_SIZE = 3

# Apply FIFO trimming
active_context = apply_fifo_window(conversation, window_size=WINDOW_SIZE)

print(f"Original History Length : {len(conversation)} messages")
print(f"Active FIFO Window Size : {len(active_context)} messages\n")

print("--- ACTIVE CONTEXT SENT TO GRANITE ---")
for idx, msg in enumerate(active_context):
    role = "SYS" if isinstance(msg, SystemMessage) else ("USER" if isinstance(msg, HumanMessage) else "BOT")
    print(f"{idx+1}. [{role}]: {msg.content}")

### In-Window vs. Out-of-Window Memory Retention

To check where FIFO succeeds and where it breaks down, we ask Granite two follow-up questions:

- **In-Window:** A query about airport transit (present in the last 3 pairs). Granite answers accurately.
- **Out-of-Window:** A query referencing the \$100–\$200 daily budget (dropped by the window). Granite fails to recall it.

**Key Limitation of FIFO**

FIFO permanently discards older turns. Any fact — a budget cap, an agreed constraint, a user preference — that falls outside the window is **gone forever** from the model's view. This trade-off is fundamental and cannot be fixed by tuning `window_size` alone.

This limitation is addressed by strategies covered later in this notebook like _Compaction / Summarization_, _Structured Note-Taking_ and _Filesystem as Scratchpad_

In [ ]:
# --- Part 3: Memory Retention Test ---

# 1. In-Window Test (Narita Airport info is present in the active window)
q_in = HumanMessage(content="Can you remind me what train option you suggested for Narita airport?")
test_context_in = active_context + [q_in]
res_in = llm.invoke(test_context_in)

print("=== 1. IN-WINDOW TEST ===")
print(f"User Question: {q_in.content}")
print(f"Granite      : {res_in.content}\n")

# 2. Out-of-Window Test (Budget details were dropped by the rolling window)
q_out = HumanMessage(content="Is a $250 hotel near Shinjuku within the daily budget I mentioned earlier?")
test_context_out = active_context + [q_out]
res_out = llm.invoke(test_context_out)

print("=== 2. OUT-OF-WINDOW TEST ===")
print(f"User Question: {q_out.content}")
print(f"Granite      : {res_out.content}")

Notice how Granite gave a generic guess about what is *"reasonable for Tokyo"* rather than confirming whether \$250 fits your specific budget range (\$100–\$200). Because the budget conversation occurred in turns 4 and 5 (which were trimmed **out of the context window**), Granite had zero memory of the \$100–\$200 figure — a direct consequence of FIFO's permanent-discard behaviour.

## Strategy 2 — Compaction / Summarization

**Goal:** Reduce token size without losing important facts from early in the conversation.

FIFO permanently dropped the budget turns, so Granite had no idea the user had specified a \$100–\$200 budget. Compaction fixes this by **summarizing** the dropped turns rather than discarding them.

**How it works:** We reuse the same `conversation`, `apply_fifo_window`, and `print_compact_conversation` from Strategy 1. Instead of throwing away the turns that fall outside the window, we feed them to the LLM and ask it to distill them into a short bullet-point summary. That summary is then **appended directly to the existing `SystemMessage`** — keeping one single `[SYS]` block rather than inserting a second one.

The resulting context structure sent to the LLM is:

```
[SYS]  <original persona>
       --- Summary of earlier conversation ---
       • Season: Spring or Autumn recommended
       • Budget: $100–$200/day mid-range
       • Neighbourhoods: Shibuya, Shinjuku, Asakusa, Ginza
[USER] most recent pair N-1 question   <- pair-wise, same as Strategy 1
[BOT]  most recent pair N-1 answer
[USER] most recent pair N question
[BOT]  most recent pair N answer
```

**What else can go into the SystemMessage?**  

The summary is appended after the persona, but you can extend this section to include:
- **User preferences** captured across sessions (language, tone, output format)
- **Agreed constraints** (budget caps, deadlines, policy rules)
- **Tool usage notes** (which tools are available in this session)
- **Stateful flags** (e.g. `task_status: in_progress`, `escalation_required: false`)  

Anything you want the model to treat as persistent background fact belongs here.

**Strategy 7 Note — Semantic Compression vs. Natural Language Summarization**  

While standard compaction summarizes text into natural language bullets or prose, **semantic compression (Strategy 7)** distills conversation history into a structured schema (e.g., JSON key-value pairs). This achieves maximum token efficiency by capturing critical operational parameters (budgets, dates, statuses) directly into state, intentionally trading away conversational nuance and tone to minimize token overhead.

### Defining the Compaction Logic

The compactor:
1. Calls `apply_fifo_window(conversation, window_size=2)` (from Strategy 1) to get the recent pairs already trimmed correctly
2. Derives the dropped turns by diffing the full `conversation` against the window output
3. Asks the LLM to summarize those dropped turns into bullet points
4. Appends the summary to the `SystemMessage` content — one `[SYS]` block, no extras

In [ ]:
# =====================================================================
# Strategy 2 — Compaction / Summarization
# Reuses: conversation, apply_fifo_window, print_compact_conversation
#         (all defined in Strategy 1)
# =====================================================================

# --- Step 1: Use apply_fifo_window from Strategy 1 to get the recent pairs ---

# Retain the most recent turns and separate the system prompt from the chat history
COMPACT_PAIRS_TO_KEEP = 2
windowed = apply_fifo_window(conversation, window_size=COMPACT_PAIRS_TO_KEEP)

original_sys = windowed[0]
recent_pairs_msgs = windowed[1:]

# Identify dropped turns: everything between the SystemMessage and the recent pairs
# conversation[0] = SystemMessage, conversation[1:] = all chat turns
all_chat = conversation[1:]
dropped_turns = all_chat[: len(all_chat) - len(recent_pairs_msgs)]

# --- Step 2: Format dropped turns for the summarization prompt ---
formatted_dropped = ""
for msg in dropped_turns:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    formatted_dropped += f"{role}: {msg.content}\n"

# --- Step 3: Ask the LLM to distill dropped turns into key facts ---
summary_prompt = [
    SystemMessage(content="You are a precise conversation summarizer."),
    HumanMessage(content=(
        "Summarize the key decisions, constraints, user preferences, and facts from "
        "this conversation history into 3-4 bullet points. Be concise.\n\n"
        f"Conversation History:\n{formatted_dropped}"
    ))
]
summary_text = llm.invoke(summary_prompt).content.strip()

# --- Step 4: Append summary to the existing SystemMessage — no second SYS block ---
merged_sys = SystemMessage(
    content=(
        original_sys.content
        + "\n\n--- Summary of earlier conversation ---\n"
        + summary_text
    )
)

# --- Step 5: Assemble compacted context ---
compacted_context = [merged_sys] + recent_pairs_msgs

print(f"Original History Length : {len(conversation)} messages")
print(f"Compacted Context Size  : {len(compacted_context)} messages")
print_compact_conversation(compacted_context, title="COMPACTED CONTEXT SENT TO GRANITE")

### Testing Memory Retention After Compaction

While using FIFO we saw that it forgot the \$100–\$200 budget. We now reuse the **exact same `q_out` question** from previous examples against `compacted_context` to show whether the summary rescued that dropped fact.

In [ ]:
# =====================================================================
# Strategy 2 — Verify Memory (reuses q_out from Strategy 1)
# =====================================================================

# q_out is already defined in Strategy 1:
# q_out = HumanMessage(content="Is a $250 hotel near Shinjuku within the daily budget I mentioned earlier?")
response = llm.invoke(compacted_context + [q_out])

print("=== STRATEGY 1 (FIFO) — budget question result ===")
print(f"Granite: {res_out.content}\n")

print("=== STRATEGY 2 (COMPACTION) — same question, compacted context ===")
print(f"User Question: {q_out.content}")
print(f"Granite      : {response.content}")

As you run the output, you can see, FIFO had no memory of the budget at all, while Compaction recalled it from the summary appended to `[SYS]`.

The final context was a clean `[SYS + summary] → [USER][BOT] → [USER][BOT]` structure — pair-wise retained, single system block, no orphaned messages.

**Compaction is still lossy.** A weaker or hallucinating summarizer could drop or distort facts. For hard constraints (budgets, deadlines, policy rules) consider also writing them to a structured note — this is demonstrated in _Strategy of Structured Note-Taking_ which is explained further in this notebook.

## Strategy 3 — Dynamic Tool Selection

**Goal:** Prevent tool bloat and context confusion by dynamically retrieving and binding only relevant executable tools to the LLM.

**How it works:** Having 10+ tool schemas bound to an agent wastes tokens and degrades selection accuracy. Here, we define real, callable tools, index their descriptions into a vector space, retrieve the top $K$ matching tool objects, and use `llm.bind_tools()` to make them natively executable by the LLM.

This strategy is covered in depth in a dedicated recipe. refer to:

**[ToolRAG Agent](../ToolRAG/ToolRAG_Agent.ipynb)** — a complete, runnable walkthrough of Dynamic Tool Selection using IBM Granite and watsonx.ai. It covers:

- Defining a registry of callable LangChain tools
- Indexing tool descriptions into a Chroma vector store
- Retrieving the top-K most relevant tools at query time using semantic similarity
- Dynamically binding only the retrieved tools to the LLM via `llm.bind_tools()`
- Assembling the full retrieval-augmented agent graph with LangGraph

**Key Takeaway:** Binding fewer, semantically matched tools reduces prompt token overhead and narrows the model's action space — both of which improve correct tool selection. It does not guarantee perfect multi-tool planning on every run, but it significantly improves the odds compared to binding all tools at once.

## Strategy 4 — Context Pruning

**Goal:** Eliminate noisy, irrelevant log entries and distractors before they poison the agent's context window.

**How it works:** When agents query monitoring tools or APIs, they receive walls of raw text containing high-noise data. Unpruned context causes Context Poisoning, leading the model to hallucinate or summarize irrelevant distractors.

Unlike summarization (which rephrases text), pruning acts as a **surgical filter** — discarding distractor lines completely while preserving the exact raw text of relevant entries.

**This example uses LLM-based pruning.** The raw logs and the target query are sent to the LLM with a strict "log filter" system prompt. The LLM reads every line and returns only those that directly relate to the query — no rephrasing, no summarizing, just selective retention of exact original text. The key steps are:

1. Raw tool output (noisy logs) + target query → sent to `llm_filter_logs()`
2. LLM returns only the relevant log line(s) as `filtered_logs`
3. Agent is invoked twice — once with `raw_tool_output` (noisy) and once with `filtered_logs` (clean) — so we can compare the responses side by side

**Note:** The `keyword_score` and `simple_tokenize` helper functions (defined in [Strategy 1](#strategy-1--fifo--rolling-window)) can be used for lightweight programmatic pruning — see the section at the end of this strategy for when to choose programmatic vs. LLM-based pruning.


### Setting Up Noisy Tool Stream with a Target Signal

In [ ]:
# =====================================================================
# Strategy 4 — Context Pruning (Part 1: Realistic Noisy Log Stream)
# =====================================================================

# Simulated raw monitoring log output containing heavy noise + 1 critical signal
raw_tool_output = """
[13:58:01] SYSTEM: Routine database backup completed. 0 errors logged.
[13:59:12] DEPLOY: Automated frontend asset sync to CDN region US-East finished.
[14:01:05] ALERT: Customer checkout API returned 500 errors. Cause: DB connection pool exhausted (max 50/50 connections occupied) due to unindexed query on 'orders_v2' table.
[14:02:30] MEMO: Reminder - Engineering team brown-bag lunch session starts at 12:30 PM tomorrow in Room 4B.
[14:03:15] INFRA: Kubernetes node worker-node-8 autoscaled CPU from 40% to 55%.
[14:04:22] SECURITY: Automated SSL certificate renewal check succeeded for domain checkout.internal.
[14:05:00] LOG: Internal admin dashboard user logged out after 30 mins idle time.
[14:06:10] DEPLOY: CI/CD pipeline #8841 status changed to SUCCESS for microservice-billing.
""".strip()

# Target query requiring a specific answer buried in the noise
user_query = "What specific root cause caused the customer checkout failures during the 14:00 window?"

print(f"Raw Monitoring Stream Loaded ({len(raw_tool_output)} characters).")
print(f"Target Query: '{user_query}'\n")
print("--- RAW UNPRUNED LOG CONTEXT ---")
print(raw_tool_output[:300] + "\n... [Rest of raw logs omitted] ...")

### LLM-Based Log Filter

In [ ]:
# =====================================================================
# Strategy 4 — Context Pruning: LLM-Based Log Filter
# =====================================================================

from langchain_core.messages import HumanMessage, SystemMessage

def llm_filter_logs(query: str, raw_context: str) -> str:
    """
    LLM-based pruner: sends raw logs + query to the LLM with a strict
    'log filter' prompt. Returns only the exact log lines relevant to
    the query — no rephrasing, no summarizing.
    """
    filter_prompt = [
        SystemMessage(content=(
            "You are a precise log filter. Extract ONLY the exact log lines or passages "
            "from the raw input that directly relate to or answer the user's target query.\n"
            "- Do NOT rephrase, summarize, or alter the original log text.\n"
            "- Do NOT add conversational intro or outro filler.\n"
            "- Completely remove all unrelated system logs, deploys, and announcements."
        )),
        HumanMessage(content=(
            f"TARGET QUERY: {query}\n\n"
            f"RAW LOGS TO FILTER:\n{raw_context}"
        ))
    ]

    return llm.invoke(filter_prompt).content.strip()

print("LLM log filter ready.")

### Comparing Results (Without Pruning vs. With Pruning)

In [ ]:
# =====================================================================
# Strategy 4 — Context Pruning: Noisy vs. Clean comparison
# =====================================================================

# 1. Run LLM-based filter to isolate the relevant log line(s)
filtered_logs = llm_filter_logs(user_query, raw_tool_output)

# 2. Noisy prompt — agent sees the full unfiltered log stream
noisy_prompt = [
    SystemMessage(content="You are a SRE Incident Analyst. Answer the user's query directly based on the context."),
    HumanMessage(content=f"QUERY: {user_query}\n\nCONTEXT:\n{raw_tool_output}")
]
noisy_response = llm.invoke(noisy_prompt)

# 3. Clean prompt — agent sees only the LLM-filtered relevant lines
clean_prompt = [
    SystemMessage(content="You are a SRE Incident Analyst. Answer the user's query directly based on the context."),
    HumanMessage(content=f"QUERY: {user_query}\n\nCONTEXT:\n{filtered_logs}")
]
clean_response = llm.invoke(clean_prompt)


# --- DISPLAY COMPARISON METRICS & RESPONSES ---
print("=" * 70)
print(" 1. CONTEXT FOOTPRINT METRICS")
print("=" * 70)
print(f"Raw (noisy) context   : {len(raw_tool_output)} characters")
print(f"Filtered (clean) logs : {len(filtered_logs)} characters")
print(f"Noise removed         : {((len(raw_tool_output) - len(filtered_logs)) / len(raw_tool_output)) * 100:.1f}%")

print("\n" + "=" * 70)
print(" 2. FILTERED LOGS PASSED TO AGENT")
print("=" * 70)
print(filtered_logs)

print("\n" + "=" * 70)
print(" 3. AGENT RESPONSE COMPARISON")
print("=" * 70)
print(f"--- NOISY response (full raw log) ---\n{noisy_response.content}\n")
print(f"--- CLEAN response (filtered logs) ---\n{clean_response.content}")

The **CONTEXT FOOTPRINT METRICS** show the LLM filter stripped ~78% of the log clutter (CDN syncs, lunch memos, SSL renewals) and isolated exactly the one relevant alert line.

The **noisy response** shows the classic poisoning effect — the model got distracted and falsely attributed the checkout failures to the CDN asset sync that appeared nearby in the log stream.

The **clean response** grounds its answer in the real signal: an unindexed query on `orders_v2` exhausting the connection pool. The analysis is still generated, but it is now based on the correct evidence.

Context Pruning is not just about saving tokens — it directly reduces the risk of false correlations and hallucinated root causes.

### LLM-Based vs. Programmatic Pruning

This example used **LLM-based pruning**: the model itself decided which lines were relevant. That is powerful but has a cost — it requires a full LLM call just to filter, before the actual agent call.

*Programmatic pruning* skips the extra LLM call entirely by applying deterministic rules in Python. The `keyword_score` and `simple_tokenize` helpers defined in Strategy 1 are a lightweight example of this approach.

```python
# Example: programmatic pruning using keyword_score from Strategy 1
def programmatic_filter_logs(query: str, raw_logs: str, min_score: int = 1) -> str:
    lines = raw_logs.strip().splitlines()
    return "\n".join(line for line in lines if keyword_score(query, line) >= min_score)
```

Other programmatic approaches include *Regex / prefix filtering*, *Timestamp windowing*, *Structured field extraction*. We can choose the pruning technique based on the requirement.


| Pruning type | Best for | Trade-off |
| :--- | :--- | :--- |
| **LLM-based** | Unstructured free-text logs; complex relevance judgment; when accuracy matters more than latency | Extra LLM call adds cost and latency |
| **Regex / prefix** | Structured logs with known noisy prefixes (e.g. `DEPLOY:`, `MEMO:`) | Brittle if log format changes |
| **Keyword score** | Semi-structured logs; cheap pre-filter before LLM; latency-sensitive pipelines | Misses semantic matches (synonyms, paraphrases) |
| **Timestamp window** | Time-bounded incident investigations; event correlation | Irrelevant lines within the window still pass through |
| **Field extraction** | JSON/CSV tool outputs with known schema | Requires structured input; not suitable for free-text |

## Strategy 5 & 6 — Structured Note-Taking + Filesystem as Scratchpad

Both strategies solve the same core problem — offloading important facts out of the raw chat history so the model's active context stays lean. The extraction logic is identical. The **only fundamental difference is where the notes live**:

| Metricmeasurement metrics| Strategy 5 — Structured Note-Taking | Strategy 6 — Filesystem as Scratchpad |
| :--- | :--- | :--- |
| **Storage** | In-memory Python dict / LangGraph state | Files on disk (`.json`, `.md`) |
| **Lifetime** | Current agent run only | Survives process restarts and session boundaries |
| **Accessible by** | Same graph run | Any agent, process, or human that can read the file |
| **Best for** | Single-session multi-step workflows | Multi-session tasks, pause/resume, audit trails, multi-agent handoffs |
| **Cost** | Zero I/O overhead | Disk read/write (negligible for most workloads) |

**The natural progression:** extract structured notes into memory (Strategy 5) → then persist them to disk (Strategy 6) → read back selectively on the next turn. Showing this as one continuous example makes the relationship explicit and avoids duplication.

**Why save to `.json` vs `.md`?**
- **`.json`** — machine-readable; ideal for structured facts (budgets, IDs, flags) that another agent step or tool needs to parse programmatically.
- **`.md`** — human-readable; ideal for checklists, plans, and status logs that need to be audited or handed off to a human.

You can write both at once — the `.json` feeds downstream automation, the `.md` feeds human review.

*Security note: Always scope file operations to an isolated workspace folder (e.g., `./tmp/agent_workspace`) to prevent unauthorized access to system files.*

### Step 1 — Define State Schema and Raw Findings

The `NoteAgentState` holds both the in-memory structured notes (Strategy 5) and tracks the scratchpad path for disk persistence (Strategy 6). We use the same RFP evaluation scenario throughout.

In [ ]:
# =====================================================================
# Strategy 5 & 6 — Step 1: State schema and raw findings
# =====================================================================

import json
import shutil
from pathlib import Path
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

# LangGraph state: holds in-memory notes (Strategy 5) and
# the path to the persisted scratchpad dir (Strategy 6)
class NoteAgentState(TypedDict):
    raw_findings: list[str]          # Temporary raw text inputs
    structured_notes: dict[str, str] # In-memory extracted key-value facts (Strategy 5)
    scratchpad_dir: str              # Path to on-disk scratchpad folder (Strategy 6)
    final_summary: str               # Output synthesized from notes

# Simulated multi-step RFP evaluation findings
rfp_research_findings = [
    "Legal analysis: The vendor agreement mandates strict EU GDPR compliance with data residency in Frankfurt.",
    "Finance review: Maximum annual software license budget is capped at $120,000 USD with net-30 payment terms.",
    "Security audit: Vendor must hold active SOC2 Type II certification and support SSO via SAML 2.0.",
]

# Set up sandboxed scratchpad directory (Strategy 6)
workspace_dir = Path("tmp/agent_scratchpad")
if workspace_dir.exists():
    shutil.rmtree(workspace_dir)
workspace_dir.mkdir(parents=True, exist_ok=True)

print(f"State schema ready. Scratchpad: '{workspace_dir.resolve()}'")

### Step 2 — Extract Notes into Memory (Strategy 5) and Persist to Disk (Strategy 6)

The `extract_and_persist_node` does both jobs in one pass:
1. Sends raw findings to the LLM → extracts structured key-value facts into `state["structured_notes"]` **(Strategy 5 — in-memory)**
2. Immediately writes those same notes to `rfp_notes.json` and `rfp_notes.md` **(Strategy 6 — on-disk)**

This way the notes are simultaneously available to the current run's next node **and** durable across restarts or handoffs to another agent.

In [ ]:
# =====================================================================
# Strategy 5 & 6 — Step 2: Extract notes in-memory + persist to disk
# =====================================================================

def extract_and_persist_node(state: NoteAgentState) -> NoteAgentState:
    """
    Strategy 5: LLM extracts structured key-value notes from raw findings
                into state['structured_notes'] (in-memory dict).
    Strategy 6: Those same notes are written to rfp_notes.json (.json for
                machine use) and rfp_notes.md (.md for human audit/handoff).
    """
    raw_findings = state.get("raw_findings", [])
    scratchpad = Path(state.get("scratchpad_dir", "tmp/agent_scratchpad"))

    # --- Strategy 5: Extract structured notes via LLM ---
    prompt = [
        SystemMessage(content=(
            "You are a structured note-taking assistant. Extract core facts from the provided text.\n"
            "Return a valid JSON object where keys are topic categories "
            "(e.g. 'compliance_requirement', 'budget_cap') and values are concise factual statements.\n"
            "Output ONLY valid JSON."
        )),
        HumanMessage(content="Raw findings to extract:\n" + "\n".join(raw_findings))
    ]
    response = llm.invoke(prompt)

    try:
        clean = response.content.strip().replace("```json", "").replace("```", "").strip()
        notes = json.loads(clean)
    except Exception:
        notes = {"extracted_summary": response.content.strip()}

    # --- Strategy 6: Persist notes to disk immediately ---
    # .json — machine-readable, for downstream agent steps or tools
    json_file = scratchpad / "rfp_notes.json"
    json_file.write_text(json.dumps(notes, indent=2), encoding="utf-8")

    # .md — human-readable, for audit trail or human handoff
    md_lines = ["# RFP Evaluation Notes\n"]
    for key, value in notes.items():
        md_lines.append(f"## {key.replace('_', ' ').title()}\n{value}\n")
    md_file = scratchpad / "rfp_notes.md"
    md_file.write_text("\n".join(md_lines), encoding="utf-8")

    print(f"[Strategy 5] Notes extracted into memory : {list(notes.keys())}")
    print(f"[Strategy 6] Notes persisted to disk:")
    print(f"  ✓ {json_file.name} ({json_file.stat().st_size} bytes) — machine-readable")
    print(f"  ✓ {md_file.name}  ({md_file.stat().st_size} bytes)  — human-readable")

    return {**state, "structured_notes": notes}

print("extract_and_persist_node defined.")

### Step 3 — Synthesize from In-Memory Notes, then Verify Disk Persistence

The synthesis node uses only `state["structured_notes"]` (the in-memory dict) — no raw findings, no full chat history in the prompt.

After synthesis we demonstrate Strategy 6's durability: we reload the notes from `rfp_notes.json` to prove the facts survive beyond the current in-memory state.

In [ ]:
# =====================================================================
# Strategy 5 & 6 — Step 3: Synthesize from notes + verify disk reload
# =====================================================================

def synthesize_node(state: NoteAgentState) -> NoteAgentState:
    notes = state.get("structured_notes", {})
    prompt = [
        SystemMessage(content="You are an executive summary writer. Synthesize the structured notes into a concise RFP evaluation brief."),
        HumanMessage(content=f"STRUCTURED NOTES:\n{json.dumps(notes, indent=2)}")
    ]
    response = llm.invoke(prompt)
    return {**state, "final_summary": response.content.strip()}


# Build and run the LangGraph workflow
workflow = StateGraph(NoteAgentState)

# Nodes
workflow.add_node("extract_and_persist", extract_and_persist_node)
workflow.add_node("synthesize", synthesize_node)

# Edges
workflow.add_edge(START, "extract_and_persist")
workflow.add_edge("extract_and_persist", "synthesize")
workflow.add_edge("synthesize", END)

# Compile
note_graph = workflow.compile()

result = note_graph.invoke({
    "raw_findings": rfp_research_findings,
    "structured_notes": {},
    "scratchpad_dir": str(workspace_dir),
    "final_summary": ""
})

print("\n" + "=" * 70)
print(" FINAL EXECUTIVE BRIEF (synthesized from in-memory notes)")
print("=" * 70)
print(result["final_summary"])

# --- Strategy 6: Demonstrate disk durability ---
# Simulate a new session: reload notes from disk (in-memory state is gone)
print("\n" + "=" * 70)
print(" DISK DURABILITY CHECK (reload rfp_notes.json from scratchpad)")
print("=" * 70)
reloaded_notes = json.loads((workspace_dir / "rfp_notes.json").read_text(encoding="utf-8"))
print("Notes reloaded from disk successfully:")
print(json.dumps(reloaded_notes, indent=2))

The output shows both strategies working together in one pipeline:

- **Strategy 5 (in-memory):** The LLM extracted three key constraints (`compliance_requirement`, `budget_cap`, `security_requirement`) from three paragraphs of raw text. The synthesis node composed a full executive brief using only those structured facts — no raw findings, no chat history in the prompt.

- **Strategy 6 (on-disk):** The same notes were written to `rfp_notes.json` (machine-readable, for downstream automation) and `rfp_notes.md` (human-readable, for audit or handoff). The disk durability check then reloaded the JSON with no in-memory state — proving the facts survive a process restart or session boundary.

### Choose the stretegy as per the use case

| Use case | Recommended approach |
| :--- | :--- |
| Single-session workflow; facts only needed within this run | Strategy 5 — in-memory dict / LangGraph state |
| Multi-step pipeline that may be paused and resumed later | Strategy 6 — persist to `.json` / `.md` scratchpad |
| Multi-agent handoff (Agent A extracts, Agent B synthesizes) | Strategy 6 — shared filesystem or object store |
| Audit trail required (compliance, legal, incident post-mortem) | Strategy 6 — `.md` for human reviewers, `.json` for tooling |
| Hard constraints (budgets, SLAs, deadlines) that must survive crashes | Strategy 6 — write to disk immediately after extraction |
| Low-latency pipelines where disk I/O overhead matters | Strategy 5 — stay in memory, flush to disk only at checkpoints |


**Strategy 8 Note — Scaling from Note-Taking to Vector Retrieval (RAG)**  
While structured note-taking and scratchpads work well for short-to-medium session state, scaling to large external corpora requires 

**RAG / Vector Retrieval (Strategy 8)**. Instead of loading full documents into the prompt, RAG dynamically retrieves only the top-$K$ semantic chunks via vector similarity, avoiding the "Lost-in-the-Middle" phenomenon and reducing context bloat.
 
For complete implementations with IBM Granite, check out these dedicated recipes:
- [RAG with LangChain](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/RAG/RAG_with_Langchain.ipynb) — Foundational document chunking, vector indexing, and retrieval.
- [Agentic RAG with LangGraph](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/AI-Agents/Agentic_RAG.ipynb) — Dynamic retrieval where the agent decides when and what to retrieve.

## Strategy 9 — Sub-agent Architectures (Context Quarantine)

**Goal:** Eliminate Context Clash and multi-domain confusion by delegating sub-tasks to isolated worker agents governed by an Orchestrator and Lead Resolver.

**How it works:**

- Orchestrator: Dynamically decomposes a complex prompt and delegates independent research directives to specialized worker agents.
- Context Quarantine: Each worker executes inside its own isolated context window. Their raw conversation threads never mix.
- Conflict Resolution: The Lead Agent reviews worker outputs specifically to detect and resolve domain conflicts (e.g., Finance budget vs. Security compliance requirements) to make a unified decision.


This strategy is covered in depth across several dedicated recipes in the **Granite Snack Cookbook**. Refer to the recipes that best match your use cases:

| Recipe | What it covers | Best for |
| :--- | :--- | :--- |
| [Understanding ReAct](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/AI-Agents/Understanding_ReAct.ipynb) | The Reasoning + Acting loop that underpins most single-agent and multi-agent patterns; trace how an agent decides when to call a tool vs. produce a final answer | Learning the foundational reasoning pattern before building multi-agent systems |
| [Agentic RAG](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/AI-Agents/Agentic_RAG.ipynb) | RAG inside a LangGraph agent loop — agent decides when to retrieve, isolating retrieval context from the main reasoning thread | Agents that must query a knowledge base without flooding the primary context window |
| [GitHub Agent](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/AI-Agents/github_agent.ipynb) | A tool-calling agent that delegates GitHub API tasks to isolated tool sub-calls, keeping each tool's raw output out of the main reasoning context | Real-world example of context quarantine via tool isolation in a developer workflow |
| [Travel Planner Agent](https://github.com/ibm-granite-community/granite-snack-cookbook/blob/main/recipes/AI-Agents/travel_planner_agent.ipynb) | Multi-step planning agent that decomposes a complex travel request into parallel sub-tasks, each executed in its own context thread before synthesis | End-to-end orchestrator + worker pattern with context-isolated sub-tasks |

**Key Takeaway:** Sub-agent architectures isolate each worker's context thread so that Legal, Security, and Finance concerns never blur together during intermediate reasoning. The lead resolver then synthesizes the quarantined outputs into a single decision — dramatically reducing Context Clash and multi-domain confusion compared to a single all-in-one agent.

## Summary & Best Practices

Managing context effectively is the single most critical factor in building reliable, cost-effective LLM agents.

Every agent architecture has unique constraints, and there is no one-size-fits-all approach. Context optimization should always be driven by rigorous requirement analysis and empirical evaluation of observed agent behavior. As a best practice, **start with the simplest pattern** (such as basic FIFO or static prompt sizing) and incrementally layer in advanced strategies only when supported by measurable evaluation benchmarks.

 ### How to select the correct stretegy?:

- For simple chat applications, use FIFO (Strategy 1) or Compaction (Strategy 2).
- For agents with 10+ tools, use Dynamic Tool Selection (Strategy 3) to prevent tool bloat.
- For noisy tool/API outputs use Context Pruning (Strategy 4) to remove junk text before main agent execution.
- For long, multi-step workflows combine Structured Note-Taking + Filesystem Scratchpad (Strategy 5 & 6).
- For dense multi-turn dialogues, use Semantic Compression (Strategy 7) to reduce token overhead.
- For large document repositories, use RAG (Strategy 8).
- For complex multi-domain problems, use Sub-agent Architectures (Strategy 9) to quarantine context threads. 

## Conclusion

This notebook demonstrated all nine context management strategies using IBM Granite model calls on watsonx.ai.

Key themes across all strategies:

- **Context is not free.** Every token influences the model's output. Treat your context budget like memory budget.
- **Know your failure mode.** Poisoning, Distraction, Confusion, and Clash each require a different mitigation — the intro table maps every strategy to the failure modes it addresses.
- **Summarization loses information.** For hard constraints (budgets, deadlines, SLAs), offloading to structured notes or a filesystem scratchpad (Strategy 5 & 6) is safer than relying on compaction alone.
- **Separate information gathering from decision making.** Sub-agents work well for the former (quarantined domain analysis); a single resolver works better for the latter (final synthesis).
- **Benchmarks understate the problem.** Test at your actual context lengths with actual distractors — short-context benchmarks rarely surface Lost-in-the-Middle or Context Clash failures.
- **Structure your context.** Typed LangGraph state (as used in Strategy 5 & 6) makes pruning, summarization, and selective retrieval dramatically easier to reason about and debug.

### Where to go next

- [Granite Snack Cookbook ](https://github.com/ibm-granite-community/granite-snack-cookbook) — Explore various recipies with Granite models. 